# Builder Model

## Overview

The Builder Model is responsible for generating and modifying code based on instructions provided by the Architecture or Main Agent. Its primary role is implementation: creating new files, updating existing ones, and translating high-level specifications into working, production-ready code.

In the orchestrator, this model functions as the execution engine — it does not make architectural decisions or evaluate code quality. Instead, it focuses on writing complete, correct, and idiomatic code while following the exact structure and requirements defined by upstream agents. All files are returned in full, ensuring downstream tools (like the Review Model) can operate on complete source files.


## Example Usage

### System Instruction

In [8]:
from system_instruction import getBuilderSystemInsturction
system_instructon = getBuilderSystemInsturction()

### Output

In [9]:
from openai import OpenAI
import json
import os

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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" 
)

# TODO: change the model to deepseek 
response = client.chat.completions.create(
    model="openai/gpt-oss-20b:free",
    messages=[
        {"role": "system", "content": system_instructon},
        {
            "role": "user",
            "content": json.dumps({
                "context": "Implement the VSCode helloWorld command",
                "tasks": [
                    "Create extension entrypoint",
                    "Implement helloWorld command"
                ]
            })
        }
    ]
)

raw_output = response.choices[0].message.content
print(raw_output)

{
  "files": [
    {
      "path": "src/extension.ts",
      "content": "import * as vscode from 'vscode';\n\n/**\n * This method is called when your extension is activated.\n * Your extension is activated the very first time the command is executed.\n */\nexport function activate(context: vscode.ExtensionContext) {\n    // Register the helloWorld command\n    const disposable = vscode.commands.registerCommand('extension.helloWorld', () => {\n        // Show a message box\n        vscode.window.showInformationMessage('Hello World!');\n    });\n\n    // Push the disposable to the context so that the command is correctly disposed\n    context.subscriptions.push(disposable);\n}\n\n/**\n * This method is called when your extension is deactivated.\n */\nexport function deactivate() {\n    // Nothing to clean up in this simple example.\n}\n"
    },
    {
      "path": "package.json",
      "content": "{\n  \"name\": \"hello-world\",\n  \"displayName\": \"Hello World\",\n  \"description\": \"